In [ ]:
!python --version

Python 3.12.12


In [ ]:
!nvidia-smi

Sat Feb 21 12:36:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# DIRECTORIES

In [ ]:
import os

ROOT_DIR = "/content/drive/MyDrive/ocr_vs_llm_parsing"
DATA_DIR = f"{ROOT_DIR}/data"

os.makedirs(DATA_DIR, exist_ok=True)

# REQUIREMENTS

In [ ]:
%%shell
uv pip install vllm==0.15.1
uv pip install langchain-core==1.2.14
uv pip install langchain-openai==1.1.10
uv pip install pymupdf==1.27.1

Using Python 3.12.12 environment at: /usr
Audited 1 package in 116ms
Using Python 3.12.12 environment at: /usr
Audited 1 package in 99ms
Using Python 3.12.12 environment at: /usr
Audited 1 package in 100ms
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 43ms
Prepared 1 package in 327ms
Installed 1 package in 5ms
 + pymupdf==1.27.1


In [ ]:
!pip list

Package                                  Version
---------------------------------------- -------------------
absl-py                                  1.4.0
accelerate                               1.12.0
access                                   1.1.10.post3
affine                                   2.4.0
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
aiosqlite                                0.22.1
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.18.4
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
anthropic                                0.83.0
antlr4-python3-runtime       

# VLLM SERVE

In [ ]:
import subprocess
import time
from datetime import datetime
from typing import Tuple


def deploy_docling(
    model_name: str,
    target_message: str = "Application startup complete",
    max_dep_time: int = 2400,
    port: int = 8000,
    gpu_memory_utilization: float = 0.90,
    max_model_length: int = 6144,
    max_num_seqs: int = 1,
    max_num_batched_tokens: int = 1024,
    host: str = "0.0.0.0",
) -> Tuple[bool, str]:
    served_model_name = "_".join(model_name.split("/")[-2:])

    cmd1 = """pkill -f "vllm.entrypoints.openai.api_server .*--port {port}" || true""".format(
        **{
            "port": port,
        }
    )

    cmd2 = """export TOKENIZERS_PARALLELISM=true

vllm serve \
    --model {model_name} \
    --served-model-name {served_model_name} \
    --host {host} \
    --port {port} \
    --gpu-memory-utilization {gpu_memory_utilization} \
    --max-model-len {max_model_length} \
    --max-num-seqs {max_num_seqs} \
    --max-num-batched-tokens {max_num_batched_tokens} \
    --disable-log-stats \
    --revision untied \
    --enable-prefix-caching \
    --trust-remote-code > {served_model_name}_{port}.log 2>&1 &

sleep 3; tail -n 80 {served_model_name}_{port}.log""".format(
        **{
            "model_name": model_name,
            "served_model_name": served_model_name,
            "port": port,
            "gpu_memory_utilization": gpu_memory_utilization,
            "max_model_length": max_model_length,
            "max_num_seqs": max_num_seqs,
            "max_num_batched_tokens": max_num_batched_tokens,
            "host": host,
        }
    )

    t0 = datetime.now()

    subprocess.run(cmd1, shell=True)

    time.sleep(3)

    subprocess.run(cmd2, shell=True)

    while True:
        time.sleep(5)

        with open(f"{served_model_name}_{port}.log", "r") as log_file:
            log_content = log_file.read()

        # Check if the target message is in the log content
        if target_message in log_content:
            break

        t1 = datetime.now()

        diff = t1 - t0
        if diff.seconds >= max_dep_time:
            return False, served_model_name, None

    return True, served_model_name, port

In [ ]:
_, served_model_name, port = deploy_docling(
    model_name="ibm-granite/granite-docling-258M",
    max_num_seqs=4
)

# CONVERSION

In [ ]:
import os

PDF_DIR = f"{DATA_DIR}/pdf"
PARSED_DOCUMENTS_DIR = f"{ROOT_DIR}/parser_documents/granit_docling"

os.makedirs(PARSED_DOCUMENTS_DIR, exist_ok=True)

In [ ]:
import re
from pydantic import BaseModel
from typing import List, Optional, Union


class Coordinates(BaseModel):
    x0: Union[int, float]
    y0: Union[int, float]
    x1: Union[int, float]
    y1: Union[int, float]


class LayoutElement(BaseModel):
    text: str
    id: Union[str, int]
    classification: Optional[str] = None
    coordinates: Optional[Coordinates] = None


class ParsedPage(BaseModel):
    elements: List[LayoutElement] = list()


def convert_to_markdown_coords_only(raw_text):
    lines = raw_text.split('\n')

    page = ParsedPage()

    for i, line in enumerate(lines):
        coord_header_match = re.match(r'^((?:<loc_\d+>)+)', line)
        coords_str = ""
        if coord_header_match:
            numbers = re.findall(r'\d+', coord_header_match.group(1))
            coords = numbers

        clean_text = re.sub(r'<loc_\d+>', '', line).strip()

        if not clean_text:
            continue

        if coords and len(coords)==4:
            coords = list(map(int, coords))
            page.elements.append(LayoutElement(
                **{
                    "text": clean_text,
                    "id": i,
                    "coordinates": Coordinates(
                        **{
                            "x0": coords[0] / 1000,
                            "y0": coords[1] / 1000,
                            "x1": coords[2] / 1000,
                            "y1": coords[3] / 1000,
                        }
                    )
                }
            ))
        elif coords and len(coords)==8:
            coords = list(map(int, coords))
            page.elements.append(LayoutElement(
                **{
                    "text": clean_text,
                    "id": i,
                    "coordinates": Coordinates(
                        **{
                            "x0": min(coords[::2]) / 1000,
                            "y0": min(coords[1::2]) / 1000,
                            "x1": max(coords[::2]) / 1000,
                            "y1": max(coords[1::2]) / 1000,
                        }
                    )
                }
            ))
        else:
            page.elements.append(LayoutElement(
                **{
                    "text": clean_text,
                    "id": i,
                    "coordinates": None
                }
            ))

    return page

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate(
    [
        ("user", [
            {
                "type": "image_url",
                "image_url": {"url": "data:image/png;base64,{img64}"}
            },
            {
                "type": "text",
                "text": "Convert this page to docling."
            }
        ])
    ]
)

model = ChatOpenAI(
    model=served_model_name,
    api_key="None",
    base_url="http://127.0.0.1:8000/v1",
    max_tokens=4096,
    temperature=0.0,
)

chain = prompt | model

In [ ]:
from typing import Any, Dict
from uuid import UUID
from tqdm.auto import tqdm
from langchain_core.callbacks import BaseCallbackHandler

class BatchCallback(BaseCallbackHandler):
	def __init__(self, total: int):
		super().__init__()
		self.count = 0
		self.progress_bar = tqdm(total=total) # define a progress bar

	# Override on_llm_end method. This is called after every response from LLM
	def on_llm_end(self, response: Any, *, run_id: UUID, parent_run_id: UUID | None = None, **kwargs: Any) -> Any:
		self.count += 1
		self.progress_bar.update(1)

	def __enter__(self):
		self.progress_bar.__enter__()
		return self

	def __exit__(self, exc_type, exc_value, exc_traceback):
		self.progress_bar.__exit__(exc_type, exc_value, exc_traceback)

	def __del__(self):
		self.progress_bar.__del__()

In [ ]:
import glob
import fitz
import json
import base64
import io
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

pdf_paths = glob.glob(f"{PDF_DIR}/*.pdf")[:50]
target_size = 1024

for path in tqdm(pdf_paths):
    doc = fitz.open(path)

    inputs = list()

    for page_num in range(2, len(doc)):
        page = doc.load_page(page_num)
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img_data = pix.tobytes("png")
        img = Image.open(io.BytesIO(img_data))
        img.thumbnail((target_size, target_size))

        buffered = BytesIO()
        img.save(buffered, format="PNG")
        img_str = base64.b64encode(buffered.getvalue())
        img64 = img_str.decode('utf-8')

        inputs.append(
            {
                "img64": img64
            }
        )

    # use this if you want to see a progress bar with LangChain
    # with BatchCallback(len(inputs)) as cb: # init callback
    #     outputs = chain.batch(inputs, config={"callbacks": [cb], "batch_size": 4})

    outputs = chain.batch(inputs, config={"batch_size": 4})

    parsed_doc = dict()
    for page_n, output in enumerate(outputs, 1):
        parsed_doc[page_n] = convert_to_markdown_coords_only(output.content).model_dump()

    # ---- save ----

    filename = path.split("/")[-1].replace(".pdf", "")
    with open(f"{PARSED_DOCUMENTS_DIR}/{filename}.json", "w") as f:
        json.dump(parsed_doc, f)

  0%|          | 0/50 [00:00<?, ?it/s]